In [1]:
import pandas as pd

In [4]:
#Гружу файл 
df = pd.read_csv('Данные для тестового задания - Данные об аудитории.csv')
df['date'] = pd.to_datetime(df['date'])

In [5]:
# 1. MAU
mau = df['user_id'].nunique()
print(f"1. MAU: {mau}")

1. MAU: 7639


In [6]:
# 2. DAU
dau = df.groupby('date')['user_id'].nunique().mean()
print(f"2. DAU: {round(dau)}")

2. DAU: 560


In [7]:
# 3. Retention 1-го дня (кохорта 1 ноября)
nov1_users = set(df[df['date'] == '2023-11-01']['user_id'])
nov2_users = set(df[df['date'] == '2023-11-02']['user_id'])
retention_d1 = len(nov1_users.intersection(nov2_users)) / len(nov1_users) * 100
print(f"3. Retention Day 1: {retention_d1:.1f}%")

3. Retention Day 1: 26.6%


In [8]:
# 5. Конверсия в просмотр объявлений
ad_viewers = df[df['view_adverts'] > 0]['user_id'].nunique()
conversion = (ad_viewers / mau) * 100
print(f"5. Conversion view_adverts: {conversion:.1f}%")

5. Conversion view_adverts: 46.3%


In [9]:
# 6. Среднее количество просмотренных объявлений на пользователя
avg_ads = df.groupby('user_id')['view_adverts'].sum().mean()
print(f"6. Avg adverts per user: {avg_ads:.1f}")

6. Avg adverts per user: 2.9


In [12]:
# 7. Расчет NPS

import scipy.stats as stats
import numpy as np

promoters, detractors, total_nps = 1200, 500, 2000
nps = ((promoters - detractors) / total_nps) * 100
print(f"7. NPS: {nps:.0f}%")

7. NPS: 35%


In [14]:
import pandas as pd
import scipy.stats as stats
df_ab = pd.read_csv('Данные для тестового задания - Данные АБ тестов.csv')
df_listers = pd.read_csv('Данные для тестового задания - Листеры.csv')

#8. АБ-тесты
print("АБ-тесты")
for exp in sorted(df_ab['experiment_num'].unique()):
    sub = df_ab[df_ab['experiment_num'] == exp]
    control = sub[sub['experiment_group'] == 'control']['revenue']
    test = sub[sub['experiment_group'] == 'test']['revenue']
    
    t_stat, p_val_t = stats.ttest_ind(control, test)
    mw_stat, p_val_mw = stats.mannwhitneyu(control, test)
    
    print(f"Эксперимент {exp}:")
    print(f"  ARPU Control: {control.mean():.2f}, ARPU Test: {test.mean():.2f}")
    print(f"  p-value (t-test): {p_val_t:.4f}")
    print(f"  p-value (Mann-Whitney): {p_val_mw:.4f}\n")

АБ-тесты
Эксперимент 1:
  ARPU Control: 722.46, ARPU Test: 665.74
  p-value (t-test): 0.6888
  p-value (Mann-Whitney): 0.7960

Эксперимент 2:
  ARPU Control: 704.65, ARPU Test: 332.93
  p-value (t-test): 0.0010
  p-value (Mann-Whitney): 0.0085

Эксперимент 3:
  ARPU Control: 663.21, ARPU Test: 998.67
  p-value (t-test): 0.0617
  p-value (Mann-Whitney): 0.0010



In [15]:
# 9. Ср. доход на пользователя
rev_per_user = df_listers.groupby('user_id')['revenue'].sum().mean()
print(f"9. Средний доход на пользователя: {rev_per_user:.1f}")

9. Средний доход на пользователя: 156.5


In [16]:
# 10. Медиана возраста
median_age = df_listers['age'].median()
print(f"10. Медиана возраста: {median_age:.0f}")

10. Медиана возраста: 28


In [17]:
#18. Статистическая проверка АБ-теста (100 млн пользователей)
n_a, pay_a = 100047501, 1003
n_b, pay_b = 100001055, 1099

cr_a = pay_a / n_a
cr_b = pay_b / n_b

# Z-тест пропорций
p_pool = (pay_a + pay_b) / (n_a + n_b)
se = np.sqrt(p_pool * (1 - p_pool) * (1/n_a + 1/n_b))
z = (cr_b - cr_a) / se
p_val_18 = 2 * (1 - stats.norm.cdf(abs(z)))

print(f"18. CR_A: {cr_a:.8f}, CR_B: {cr_b:.8f}")
print(f"18. Z-score: {z:.4f}, p-value: {p_val_18:.4f}")

18. CR_A: 0.00001003, CR_B: 0.00001099
18. Z-score: 2.1046, p-value: 0.0353
